# 03 · Benchmark design: audit and answer key

1. **Audit** the LLM labels against human labels (target ≥ 90% agreement).
2. **Generate** the questions and compute exact answers from the labels with pandas.

The answer key is only as good as the labels, which is why the audit comes first.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from rlm_logistics.questions import build_questions

ROOT = Path("..")
labels = pd.read_csv(ROOT / "data/labels/labels.csv")
meta = json.loads((ROOT / "data/processed/contexts.json").read_text())
records = pd.read_csv(ROOT / "data/processed/records.csv").merge(labels, on="order_id")
print(f"{len(records):,} labeled records")

## 1. Human audit
`data/labels/audit_sheet.csv` has 200 random reviews with an LLM translation, but **not** the LLM's label (a *blind* audit),
so the human isn't nudged toward agreeing. Fill `human_label` with one of: delivery, product, seller_service, positive, other.

In [ ]:
audit = pd.read_csv(ROOT / "data/labels/audit_sheet.csv").merge(labels, on="order_id")
done = audit[audit.human_label.notna()]
if done.empty:
    print("Audit not filled in yet: fill human_label in data/labels/audit_sheet.csv, then re-run.")
else:
    agreement = (done.human_label == done.label).mean()
    print(f"agreement: {agreement:.1%} on {len(done)} reviews -> {'PASS' if agreement >= 0.9 else 'FIX THE PROMPT'}")
    display(pd.crosstab(done.human_label, done.label, rownames=["human"], colnames=["LLM"]))

Disagreements are the most useful rows to read: they show whether the label definitions are unclear or the model is wrong.

In [ ]:
if not done.empty:
    display(done[done.human_label != done.label][["english", "human_label", "label"]])

## 2. Questions and answer key
* **T1 lookup**: the complaint of one order. Work stays constant as the context grows.
* **T2 aggregation**: a percentage over all matching orders. Work grows linearly.
* **T3 pairwise**: every pair of orders meeting a condition. Work grows quadratically.
  Its scope (all orders or one state) is chosen so each answer has 3-60 pairs, which keeps answers checkable.

6 questions per tier at 32K/128K/512K, plus 2 per tier at 2M.

In [ ]:
contexts = {name: records.iloc[:m["n_records"]] for name, m in meta.items()}
qs = build_questions(contexts, {"32k": 6, "128k": 6, "512k": 6, "2m": 2})
qs.to_json(ROOT / "data/processed/questions.jsonl", orient="records", lines=True, force_ascii=False)
qs.groupby(["size", "tier"]).size().unstack()

The label mix should be similar at every size (same random order), so accuracy differences come from size, not content.

In [ ]:
pd.DataFrame({name: ctx.label.value_counts(normalize=True).round(3) for name, ctx in contexts.items()})

In [ ]:
for r in qs.groupby("tier").head(1).itertuples():
    print(f"[{r.id}] {r.question}\n  -> {str(r.answer)[:120]}\n")

## 3. Sanity check: can pandas alone answer these?
Only if it has the labels. Without them, every T2 and T3 answer needs a judgment about review text,
which is exactly the work RAG and RLM are tested on (PRD risk: *"questions answerable by pandas alone"*).

In [ ]:
print(qs[qs.tier == "T3"].answer.map(len).describe()[["min", "50%", "max"]])